# 模型接口：发送什么，又收到什么

本页学习目标：

- 读懂模型请求中的 `messages`、工具说明和生成参数。
- 区分普通文本、JSON 和工具请求。
- 保存每轮请求、原始响应、工具结果和停止原因。
- 判断模型是否实际使用了工具反馈。

本页正文直接解释已有运行。下方的真实调用开关默认关闭，阅读这些结果不需要启动模型。

[返回课程入口](../README.md) · 理解基本循环后再看。


## 1. 一次模型请求包含什么

| 内容 | 用途 |
| --- | --- |
| 模型标识 | 指定调用哪个模型；实验还需记录权重版本 |
| `messages` | 当前问题、指令、历史消息和已返回的工具结果 |
| `tools` | 模型可以请求的工具名称、用途和参数 |
| 生成参数 | 例如温度和输出长度上限 |
| 超时 | 请求最多等待多久 |

模型只会看到这一轮实际提交的信息。磁盘上存在一份文档，不代表模型知道文档内容。

普通回答返回文本；JSON 模式要求返回可解析的数据；提供工具说明则允许模型提出工具请求。这三种形式分别解决输出、解析和动作表达的问题，实际执行仍由程序负责。


## 2. 怎样记录一次完整运行

每一轮应按时间顺序记录：

1. **请求**：发给模型的完整消息、工具说明和参数。
2. **原始响应**：服务实际返回的内容。
3. **解析结果**：提取出的工具名、参数或最终回答。
4. **工具执行**：程序实际使用的参数、成功结果或错误。
5. **下一轮请求**：确认刚才的工具结果已经加入消息。
6. **结束检查**：为什么停止，任务是否完成。

如果适配器把模型输出的文本 JSON 转成工具请求，应该同时保留转换前后的内容，便于区分原生工具格式和程序适配。

保存请求时也要复制当时的数据。本项目早期曾直接保存可变的消息列表，后续追加消息影响了旧记录；修复后改为发送时深拷贝，并保留旧记录和同配置重跑结果。


## 3. 实际运行：只加强提示仍然没有读取资料

2026-10-20，Qwen3-0.6B 在 CPU 上执行了两次关于课程评测要求的任务。

第一次要求“阅读相关资料并引用”，模型回答“至少 20 道评测题在 Stage 3”，末尾带 `[L1]`。记录中没有工具请求或工具执行；正确阶段是 Stage 7。

第二次进一步要求“使用 `read_file` 读取 `course.md`”，模型仍没有请求工具，只回答该文档的来源编号是 `[L1]`，没有给出所问阶段。

两次都只有一次模型生成，程序因没有新的工具请求而返回 `model_stopped`。这个状态说明循环结束，没有说明问题已正确回答。


## 4. 实际运行：程序要求先读取资料

另一次运行增加了程序检查：没有成功读取过来源，就不能把回答当作结束。

| 顺序 | 实际事件 |
| --- | --- |
| 1 | 模型回答“来源编号不可用，无法确定” |
| 2 | 程序发现没有读取记录，拒绝结束，并明确要求读取 `course.md` |
| 3 | 模型请求 `read_file(path="course.md")` |
| 4 | 工具成功返回来源 `L1` 和课程正文，其中写明 Stage 7 要求至少 20 道评测题 |
| 5 | 下一次请求包含工具正文，模型回答“Stage 7” |

最终事实正确，但回答没有附上 `[L1]`。所以该检查只保证这次结束前发生过成功读取，并没有保证答案引用完整或每句话都得到资料支持。

这些记录使用同一个小模型，但输入和程序规则并不完全相同。它们能帮助理解恢复过程，不能用来估计一般任务上的提升幅度。


## 5. 怎样设计下一次检查

可以每次只改变一个条件，再观察行为是否变化：

| 改动 | 需要观察什么 |
| --- | --- |
| 不提供工具 | 模型是否承认无法核查本地资料 |
| 工具返回空结果 | 是否换查询或明确说明没有证据 |
| 不回传工具结果 | 下一轮是否缺少必要信息 |
| 去掉来源编号 | 引用是否仍能准确定位 |
| 缩短调用预算 | 是否明确报告未完成 |

这是建议的实验方法，本页没有给这些改动填写成绩。需要保存真实新输出，不能拿预写消息代替模型响应。


## 6. 练习与自查

选一轮工具调用，找出：模型当时看到的问题、它提出的动作、真实工具返回、包含该返回的下一轮输入。

如果这四项不能对应，就先检查记录是否完整。最后再判断答案是否正确，以及依据来自模型已有知识、程序预先提供的资料，还是模型主动请求的工具。

下一页：[ReAct 循环](03-extension-react.ipynb)，或进入 [工具、RAG 和记忆](../02-tools-rag-memory/01-rag-and-memory.ipynb)。


## 代码说明

`request` 和 `chat` 调用本地模型接口，`model_manifest` 读取模型身份，`save` 保存结果，`parse_json` 解析结构化输出。`agent` 用算术和词数工具检查真实工具交换。

默认执行只检查本地函数。`RUN_LIVE=False` 时，普通对话、JSON 生成和模型工具循环均不会运行。若要开启，先准备本地模型服务，并把 `MODEL` 改成服务中实际可用的名称；默认名称与上文历史实验的模型不同。


In [ ]:
from miniagent_course import ROOT, OUTPUT, load_lesson


In [ ]:
import json

import time

from pathlib import Path

from urllib.request import Request, build_opener, ProxyHandler

ENDPOINT = "http://127.0.0.1:11434"

<a id="impl-request"></a>

### 实现：request


In [ ]:
def request(route, body=None, timeout=90):
    data = None if body is None else json.dumps(body, ensure_ascii=False).encode("utf-8")
    req = Request(ENDPOINT + route, data, {"Content-Type": "application/json"})
    with build_opener(ProxyHandler({})).open(req, timeout=timeout) as response:
        return json.load(response)

<a id="impl-model_manifest"></a>

### 实现：model_manifest


In [ ]:
def model_manifest(model="qwen3:1.7b"):
    models = request("/api/tags")["models"]
    selected = next((m for m in models if m["name"] == model), None)
    if selected is None:
        raise RuntimeError(f"Model not installed: {model}; run ollama pull {model}")
    return {"name": selected["name"], "digest": selected["digest"],
            "details": selected["details"], "version": request("/api/version")["version"]}

<a id="impl-chat"></a>

### 实现：chat


In [ ]:
def chat(messages, *, model="qwen3:1.7b", tools=None, json_mode=False, max_output=512, timeout=90):
    body = {"model": model, "messages": messages, "stream": False, "think": False,
            "options": {"temperature": 0, "num_predict": max_output, "num_ctx": 8192}}
    if tools:
        body["tools"] = tools
    if json_mode:
        body["format"] = "json"
    start = time.monotonic()
    raw = request("/api/chat", body, timeout)
    return {"request": body, "response": raw,
            "latency_ms": round((time.monotonic()-start)*1000, 3),
            "input_tokens": raw.get("prompt_eval_count"), "output_tokens": raw.get("eval_count")}

<a id="impl-save"></a>

### 实现：save


In [ ]:
def save(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

<a id="impl-parse_json"></a>

### 实现：parse_json


In [ ]:
def parse_json(text):
    text = text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    return json.loads(text)

In [ ]:
import argparse

import json

import time

from pathlib import Path

from urllib.request import Request, build_opener, ProxyHandler

<a id="impl-calculator"></a>

### 实现：calculator


In [ ]:
def calculator(a, b, operation):
    if type(a) not in (int, float) or type(b) not in (int, float):
        raise ValueError("a/b must be numbers")
    if operation == "multiply":
        return a * b
    if operation == "add":
        return a + b
    raise ValueError("unknown operation")

<a id="impl-count_words"></a>

### 实现：count_words


In [ ]:
def count_words(text):
    if not isinstance(text, str) or len(text) > 500:
        raise ValueError("text must be <=500 characters")
    return len(text.split())

In [ ]:
REGISTRY = {"calculator": calculator, "count_words": count_words}

TOOLS = [
    {"type": "function", "function": {"name": "calculator", "description": "Calculate exact arithmetic",
     "parameters": {"type": "object", "properties": {"a": {"type": "number"}, "b": {"type": "number"},
       "operation": {"type": "string", "enum": ["add", "multiply"]}}, "required": ["a", "b", "operation"],
       "additionalProperties": False}}},
    {"type": "function", "function": {"name": "count_words", "description": "Count whitespace separated words",
     "parameters": {"type": "object", "properties": {"text": {"type": "string"}},
                    "required": ["text"], "additionalProperties": False}}},
]

<a id="impl-agent"></a>

### 实现：agent


In [ ]:
def agent(question, model="qwen3:1.7b", max_steps=6, timeout=90, tools=None):
    messages = [{"role": "system", "content": "Use tools to solve the task. Return a short factual answer."},
                {"role": "user", "content": question}]
    trace, usage = [], {"input_tokens": 0, "output_tokens": 0}
    start = time.monotonic()
    status, answer = "step_limit", ""
    for step in range(max_steps):
        remaining = timeout - (time.monotonic() - start)
        if remaining <= 0:
            status = "timeout"
            break
        payload = {"model": model, "messages": messages, "tools": TOOLS if tools is None else tools,
                   "stream": False, "think": False, "options": {"temperature": 0, "num_predict": 512}}
        request = Request("http://127.0.0.1:11434/api/chat", json.dumps(payload).encode(),
                          {"Content-Type": "application/json"})
        try:
            with build_opener(ProxyHandler({})).open(request, timeout=remaining) as response:
                raw = json.load(response)
        except Exception as exc:
            trace.append({"event": "model_error", "error": type(exc).__name__, "detail": str(exc)})
            status = "model_error"
            break
        message = raw["message"]
        if not message.get("tool_calls") and message.get("content", "").startswith("{"):
            try:
                action = json.loads(message["content"])
                if set(action) == {"name", "arguments"} and action["name"] in REGISTRY:
                    message = {"role": "assistant", "content": "", "tool_calls": [{"function": action}]}
            except (ValueError, TypeError):
                pass
        usage["input_tokens"] += raw.get("prompt_eval_count", 0)
        usage["output_tokens"] += raw.get("eval_count", 0)
        trace.append({"event": "model", "request": json.loads(json.dumps(payload)), "response": raw})
        messages.append(message)
        calls = message.get("tool_calls", [])
        if not calls:
            answer = message.get("content", "")
            status = "completed" if answer.strip() else "empty_answer"
            break
        for call in calls:
            name, arguments = call["function"]["name"], call["function"]["arguments"]
            try:
                if name not in REGISTRY:
                    raise ValueError("unregistered tool")
                spec = next(t["function"]["parameters"] for t in (TOOLS if tools is None else tools)
                            if t["function"]["name"] == name)
                if set(arguments) != set(spec["required"]):
                    raise ValueError("argument keys mismatch")
                output = {"ok": True, "data": REGISTRY[name](**arguments)}
            except Exception as exc:
                output = {"ok": False, "error": str(exc)}
            trace.append({"event": "tool", "name": name, "arguments": arguments, "result": output})
            messages.append({"role": "tool", "tool_name": name, "content": json.dumps(output)})
    return {"status": status, "answer": answer, "trace": trace, **usage,
            "latency_ms": round((time.monotonic()-start)*1000, 3)}

In [ ]:
assert calculator(12,34,"multiply") == 408
assert count_words("one two three") == 3
print(REGISTRY.keys(), calculator(12,34,"multiply"))

In [ ]:
RUN_LIVE = False
MODEL = "qwen3:1.7b"

In [ ]:
if RUN_LIVE:
    output = OUTPUT / "01-agent-loop" / str(time.time_ns())
    print(model_manifest(MODEL))
    ordinary = chat([{"role":"user", "content":"一句话解释 Agent"}], model=MODEL)
    structured = chat([{"role":"user", "content":'返回 JSON：topic 为 agent，steps 为整数 3'}], model=MODEL, json_mode=True)
    parsed = parse_json(structured["response"]["message"]["content"])
    assert parsed["topic"] == "agent" and parsed["steps"] == 3
    result = agent("Use calculator to compute 12 * 34.", model=MODEL)
    save(output / "ordinary.json", ordinary); save(output / "structured.json", structured);save(output / "loop.json", result)
    print(ordinary["response"]["message"]["content"], parsed, result["status"], result["answer"], sep="\n")
    for event in result["trace"]: print(event)
    assert result["status"] == "completed"
    assert any(e.get("event")=="tool" and e["result"].get("data")==408 for e in result["trace"])
else:
    print("真实普通对话 / JSON / 工具循环未执行；上面只验证了本地函数。")